In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
df_instantaneous_features=pd.read_parquet('../data_parquet/thermo_subsampled_daily.parquet')
df_omni_indices=pd.read_csv('../data/omniweb_data/merged_omni_indices.csv')
df_omni_magnetic_field=pd.read_parquet('../data_parquet/merged_omni_magnetic_field.parquet')
df_omni_solar_wind=pd.read_parquet('../data_parquet/merged_omni_solar_wind.parquet')
df_nrlmsise00_time_series=pd.read_csv('../data_parquet/nrlmsise00_time_series.csv')
df_soho = pd.read_parquet('../data_parquet/soho_data.parquet')

In [ ]:
dict_mean_std={}
#let's plot and save them all in a dictionary:
for df in [df_instantaneous_features, df_omni_indices, df_omni_magnetic_field, df_omni_solar_wind, df_nrlmsise00_time_series, df_soho]:
    print('Processing new dataframe')
    for feature in tqdm(df.columns):
        if feature not in ['all__dates_datetime__','source__gaps_flag__']:        
            column_data = df[feature]
            column_data=column_data[~column_data.isna()]
            mean, std = stats.norm.fit(column_data)
            #we save it:
            dict_mean_std[feature+'_mean']=mean
            dict_mean_std[feature+'_std']=std
            #we plot it
            plt.hist(column_data, bins=30, density=True, alpha=0.6, color='g')

            x = np.linspace(column_data.min(), column_data.max(), 100)
            p = stats.norm.pdf(x, mean, std)
            plt.plot(x, p, 'k', linewidth=2)
            title = f"Feature - {feature}, fit results: mean = %.2f,  std = %.2f" % (mean, std)
            plt.title(title)

            plt.show()

In [ ]:
sample_value = 30000
#int_[-inf, sample_value] pdf dx:
cumulative_prob = stats.norm.cdf(sample_value, mean, std)  # Cumulative probability
quantile_ninety_nine = stats.norm.ppf(0.99, mean, std)
quantile_zero_zero_1 = stats.norm.ppf(1-0.99, mean, std)

if sample_value > quantile_ninety_nine or sample_value < quantile_zero_zero_1:
    print(f'retrigger with {sample_value}, because cdf is: {cumulative_prob}')

In [ ]:
#with the time series, probably we can do similar things if at least a single row is outside?